# AUG-PE: Differentially Private Synthetic Text via Foundation Model APIs

This notebook walks through the AUG-PE algorithm step by step,
importing directly from the original codebase.

**Prerequisites**: Run `bash scripts/local_scripts/install_gpu.sh` first.

In [1]:
import os, sys
import numpy as np
import collections

# Ensure repo root is on the path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Working directory: {os.getcwd()}')

Working directory: /home/ubuntu/pe_llm_synthetic_gen


## 1. Privacy Accounting

Before running any experiment, verify the DP noise multipliers from the paper.

In [2]:
from src.dp_accounting import compute_sigma, compute_epsilon, compute_delta_default

# Yelp: n_priv = 1,939,290
n_priv = 1_939_290
delta = compute_delta_default(n_priv)
T = 10

print(f'Yelp: n_priv={n_priv:,}, delta={delta:.2e}, T={T}')
print()

for eps_target in [1.0, 2.0, 4.0]:
    sigma = compute_sigma(eps_target, T, delta)
    eps_check = compute_epsilon(sigma, T, delta)
    print(f'  epsilon={eps_target:.1f} => sigma={sigma:.2f} (verify: eps={eps_check:.4f})')

Yelp: n_priv=1,939,290, delta=3.56e-08, T=10

  epsilon=1.0 => sigma=15.40 (verify: eps=1.0000)
  epsilon=2.0 => sigma=8.04 (verify: eps=2.0000)
  epsilon=4.0 => sigma=4.25 (verify: eps=4.0000)


## 2. Load Private Data

Load the Yelp dataset and examine its label distribution.

In [3]:
from src.dpsda.data_loader import load_data

train_data, train_labels, label_counter, label_indexer = load_data(
    dataset='yelp',
    data_file='data/yelp/train.csv',
    num_samples=5000,  # subsample for this demo
)

print(f'Loaded {len(train_data)} private samples')
print(f'Number of label combinations: {len(label_counter)}')
print()
print('Top 10 label combinations:')
for label, count in label_counter.most_common(10):
    print(f'  {label}: {count}')

/tmp/python-venv/pe-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data_file data/yelp/train.csv
[1704050  257876  554067 ...  396709 1465262 1233310]
Loaded 4999 private samples
Number of label combinations: 50

Top 10 label combinations:
  Business Category: Restaurants	Review Stars: 5.0: 1359
  Business Category: Restaurants	Review Stars: 4.0: 693
  Business Category: Restaurants	Review Stars: 1.0: 375
  Business Category: Restaurants	Review Stars: 3.0: 348
  Business Category: Restaurants	Review Stars: 2.0: 282
  Business Category: Bars	Review Stars: 5.0: 205
  Business Category: Beauty & Spas	Review Stars: 5.0: 170
  Business Category: Shopping	Review Stars: 5.0: 136
  Business Category: Bars	Review Stars: 4.0: 132
  Business Category: Event Planning & Services	Review Stars: 5.0: 97


In [4]:
# Show a few examples
print('--- Sample private texts ---')
for i in range(3):
    print(f'\n[{train_labels[i]}]')
    print(train_data[i][:200] + '...' if len(train_data[i]) > 200 else train_data[i])

--- Sample private texts ---

[Business Category: Restaurants	Review Stars: 5.0]
We all sat , in the stuffy Music room. Awkward freshmen, yawning in the early morning class. A single boombox, stood on a stool. The music teacher walked up and pushed play.   What followed was a veri...

[Business Category: Restaurants	Review Stars: 5.0]
Best. Pancakes. Ever. So good, in fact, that we went 2 days in a row, and on the 2nd visit, waited an HOUR in line to get more. I couldn't decide if I wanted savory or sweet - an all too common conund...

[Business Category: Restaurants	Review Stars: 5.0]
Shout out to small businesses! Having to keep Christmas small this year we decided to order in from a local restaurant. I had always heard great things but had never dined there before. The food was a...


## 3. Compute Private Embeddings

Use the sentence-transformer model to embed the private data (Algorithm 1, Line 1).

In [5]:
from src.dpsda.feature_extractor import extract_features

EMBEDDING_MODEL = 'stsb-roberta-base-v2'

print(f'Computing embeddings with {EMBEDDING_MODEL}...')
private_features = extract_features(
    data=train_data,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Private embeddings shape: {private_features.shape}')

Computing embeddings with stsb-roberta-base-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4751.30it/s]
RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 5/5 [00:03<00:00,  1.33it/s]

Private embeddings shape: (4999, 768)


## 4. RANDOM_API: Generate Initial Synthetic Samples

Use GPT-2 with category/rating prompts to generate initial samples (Algorithm 1, Line 2).

In [7]:
from src.apis.hf_api import HFAPI

# Instantiate the HuggingFace GPT-2 API
# Using small batch size and few samples for this demo
api = HFAPI(
    model_type='gpt2',
    variation_type='yelp_rephrase_tone',
    use_subcategory=True,
    output_dir=None,
    seed=42,
    mlm_probability=0.5,
    length=64,
    temperature=1.4,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.0,
    do_sample=True,
    fp16=True,
    no_cuda=False,
    random_sampling_batch_size=64,
    num_beams=5,
    dry_run=False,
    variation_batch_size=64,
)
print('GPT-2 model loaded.')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1947.23it/s]

GPT-2 model loaded.


In [8]:
# Generate a small set of initial samples (Nsyn_demo samples)
Nsyn_demo = 100  # small for demo; paper uses 5000

# Scale down the label counter proportionally
demo_counter = collections.Counter()
total = sum(label_counter.values())
for label, count in label_counter.items():
    demo_count = max(1, round(count / total * Nsyn_demo))
    demo_counter[label] = demo_count

print(f'Generating {sum(demo_counter.values())} initial samples...')
initial_samples, initial_labels, sync_counter, all_prompts = api.text_random_sampling(
    num_samples=Nsyn_demo,
    prompt_counter=label_counter,
)
print(f'Generated {len(initial_samples)} initial samples')
print()
print('--- Sample generated texts ---')
for i in range(min(3, len(initial_samples))):
    print(f'\n[{initial_labels[i]}]')
    text = initial_samples[i]
    print(text[:200] + '...' if len(text) > 200 else text)

Generating 117 initial samples...


  0%|          | 0/50 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
  2%|▏         | 1/50 [00:01<01:02,  1.27s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  4%|▍         | 2/50 [00:01<00:40,  1.17it/s]Th

Generated 95 initial samples

--- Sample generated texts ---

[Business Category: Restaurants	Review Stars: 5.0]
Rating: 10 stars on Yelp.com -- 8.7 stars from Yelp users and 4.5 rating by a Hawaiian person who was familiar with our store on Instagram for many years. Great place in a great neighborhood (if you d...

[Business Category: Restaurants	Review Stars: 5.0]
Rating: 8.9 Approved: Yes (0 votes, 1 ratings) - 7 reviews Cocktails Restaurant Category Category I guess the term we like about the place doesn't really go out to much, in that sense I could think we...

[Business Category: Restaurants	Review Stars: 5.0]
Rated Score: 8.5 Overall Rating, 4 Stars Country: New Zealand Top Rated: 9.4 This is a Reviewing Report by: Ryan On January 24, 2012 at 2:12 pm


## 5. One PE Iteration

Run the core loop of AUG-PE once: embed synthetic samples, compute DP histogram,
select top samples, generate variations.

In [9]:
from src.dpsda.dp_counter import dp_nn_histogram

# Step 5a: Embed synthetic samples (K=0, self-embedding)
print('Computing synthetic embeddings...')
syn_features = extract_features(
    data=initial_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Synthetic embeddings shape: {syn_features.shape}')

Computing synthetic embeddings...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4932.67it/s]
RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:00<00:00, 16.75it/s]

Synthetic embeddings shape: (95, 768)


In [10]:
# Step 5b: DP Nearest Neighbor Histogram (one class at a time)
# For demo: use sigma=0 (non-private) to see clean signal
sigma = 0.0

private_classes = list(label_counter.keys())
all_counts = np.zeros(len(initial_samples))

current_idx = 0
for cls_label in private_classes:
    n_cls = sync_counter.get(cls_label, 0)
    if n_cls == 0:
        continue
    
    cls_syn_features = syn_features[current_idx:current_idx + n_cls]
    cls_pri_indices = label_indexer[cls_label]
    cls_pri_features = private_features[cls_pri_indices]
    
    count, clean_count = dp_nn_histogram(
        public_features=cls_syn_features,
        private_features=cls_pri_features,
        noise_multiplier=sigma,
    )
    all_counts[current_idx:current_idx + n_cls] = count
    current_idx += n_cls

print(f'Histogram: {len(all_counts)} bins, sum={all_counts.sum():.0f}')
print(f'Non-zero bins: {(all_counts > 0).sum()}')
print(f'Top 5 vote counts: {sorted(all_counts, reverse=True)[:5]}')

AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [ ]:
# Step 5c: Rank-based selection (select top samples)
# For AUG-PE with L>1: select top Nsyn/L samples, then generate L-1 variations
L = 2  # small for demo; paper uses L=7
selected_size = len(initial_samples) // L

sort_indices = np.argsort(-all_counts)
selected_indices = sort_indices[:selected_size]

selected_samples = [initial_samples[i] for i in selected_indices]
selected_labels = [initial_labels[i] for i in selected_indices]

print(f'Selected {len(selected_samples)} samples (top by histogram votes)')
print(f'Vote range of selected: [{all_counts[selected_indices[-1]]:.0f}, {all_counts[selected_indices[0]]:.0f}]')

In [ ]:
# Step 5d: VARIATION_API - generate paraphrased variations
print(f'Generating {L-1} variation(s) per selected sample...')
variations, var_labels, _, _, _ = api.text_variation(
    sequences=selected_samples,
    additional_info=selected_labels,
    num_variations_per_sequence=L - 1,
    variation_degree=0.5,
)

print(f'Variations shape: {variations.shape}')
print()
print('--- Original vs Variation ---')
for i in range(min(2, len(selected_samples))):
    print(f'\nOriginal [{selected_labels[i]}]:')
    print(f'  {selected_samples[i][:150]}')
    print(f'Variation:')
    print(f'  {variations[i, 0][:150]}')

## 6. FID Measurement

Compute the Frechet Inception Distance between the synthetic and private embedding distributions.

In [ ]:
from src.dpsda.metrics import calculate_fid

# FID of initial samples vs private data
fid_initial = calculate_fid(syn_features, private_features)
print(f'FID (initial random samples vs private): {fid_initial:.2f}')

# FID of selected samples vs private data
selected_features = extract_features(
    data=selected_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
fid_selected = calculate_fid(selected_features, private_features)
print(f'FID (after 1 PE iteration, selected): {fid_selected:.2f}')
print(f'FID improvement: {fid_initial - fid_selected:.2f}')

## 7. Next Steps

This demo showed one iteration of AUG-PE. The method produces **differentially private (DP) synthetic data** when you add Gaussian noise to the nearest-neighbor histogram; the noise level is set by `--noise_multiplier` (sigma).

- **With DP (formal privacy):** set `--noise_multiplier` to the sigma from `src.dp_accounting` for your target (ε, δ). For Yelp with ε=1 and T=10 iterations, sigma ≈ 15.34. Use fewer iterations (e.g. T=10) when running with DP.
- **Without DP (baseline):** use `--noise_multiplier 0` for a non-private baseline (faster, no formal guarantee).

Full experiment:

```bash
# Precompute full embeddings (one-time)
bash scripts/embeddings.sh --yelp

# DP run: (ε,δ)-DP with epsilon=1 (set noise_multiplier; use 10 iterations)
export CUDA_VISIBLE_DEVICES=0
# Edit scripts/hf/yelp/generate.sh: set noise=15.34, epochs=10, then:
bash scripts/hf/yelp/generate.sh

# Or run directly with DP parameters:
python src/main.py --train_data_file data/yelp/train.csv --api HFGPT --dataset yelp \
  --noise_multiplier 15.34 --model_type gpt2 --epochs 10 \
  --do_sample --length 64 --fp16 --temperature 1.4 --select_syn_mode rank \
  --num_samples_schedule 35000 --combine_divide_L 7 --init_combine_divide_L 7 \
  --variation_degree_schedule 0.5 --lookahead_degree 0 --use_subcategory \
  --feature_extractor stsb-roberta-base-v2 --feature_extractor_batch_size 1024 \
  --mlm_probability 0.5 --variation_type yelp_rephrase_tone \
  --result_folder result/yelp_dp_eps1 \
  --train_data_embeddings_file result/embeddings/stsb-roberta-base-v2/yelp_train_all.embeddings.npz \
  --random_sampling_batch_size 1024 --variation_batch_size 1024

# Evaluate downstream accuracy (edit result_folder in the script if you used a different path)
bash scripts/hf/yelp/downstream.sh
```

All source code lives under `src/`. See `src/config.py` for the paper's hyperparameter defaults and `src/dp_accounting.py` for privacy budget (sigma ↔ epsilon).